# FT11 FER2013: three neural baselines

This notebook runs the same Python data, model, training, and metric functions as the local command line. It compares the MLP, the original small CNN, and ResNet18. The majority predictor is a reference.

Training selects checkpoints with validation data. Notebook **Run All does not score the source test set**. Final test scoring is a separate command after the team freezes the model and settings.

## Start in VS Code or Colab

In VS Code, open the repository root and select `.venv`. In Colab, this notebook clones the configured GitHub repository and installs the non-PyTorch dependencies. The repository URL below comes from the configured `origin` remote. Push the reviewed code first so Colab can read this notebook and the shared pipeline.

Colab keeps its installed PyTorch and torchvision packages. The next cell checks that they can build and run the ResNet18 used by this project. It does not replace them with CPU packages. If the repository is private, set up access in Colab outside this notebook; do not paste a token here.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/AlexBearBear0319/ft11-facial-expression-recognition.git'
    ROOT = Path('/content/ft11-fer')
    if not (ROOT / '.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / 'src' / 'fer').is_dir() and (path / 'configs' / 'option1_baselines.json').is_file()), None)
    if ROOT is None:
        raise FileNotFoundError('Open this notebook from the ft11-fer repository in VS Code.')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project:', ROOT)
print('Runtime:', 'Google Colab' if IN_COLAB else 'Local Python')
print('Python:', sys.version.split()[0])

In [ ]:
import importlib.metadata
import torch
import torchvision

from src.fer.models import make_model
from src.fer.runtime import select_device

DEVICE = select_device()
probe = make_model('resnet18').to(DEVICE)
with torch.inference_mode():
    probe_output = probe(torch.zeros(1, 1, 48, 48, device=DEVICE))
assert tuple(probe_output.shape) == (1, 7), 'PyTorch and torchvision cannot run the project ResNet18.'
del probe, probe_output
print({
    'device': str(DEVICE),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'mps_available': bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()),
    'cuda_available': torch.cuda.is_available(),
})

## Get FER2013 and build the shared split

The configured source is `msambare/fer2013/versions/1`. Check its access terms first. If the local `data/fer2013/train` and `test` folders are missing, KaggleHub downloads that version into the same project folder. Complete any Kaggle access setup outside this notebook. Do not paste an access token into a cell.

In [ ]:
import json

CONFIG_PATH = ROOT / 'configs' / 'option1_baselines.json'
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
DATA_ROOT = ROOT / CONFIG['source_root']
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

if not (TRAIN_DIR.is_dir() and TEST_DIR.is_dir()):
    import kagglehub
    downloaded_path = kagglehub.dataset_download(CONFIG['source_id'], output_dir=str(DATA_ROOT))
    print('KaggleHub folder:', downloaded_path)
if not (TRAIN_DIR.is_dir() and TEST_DIR.is_dir()):
    raise FileNotFoundError(f'Expected train/ and test/ under {DATA_ROOT}. Check Kaggle access and the downloaded folder.')

from src.fer.data import build_manifest_from_config

MANIFEST_PATH = ROOT / CONFIG['manifest_path']
manifest = build_manifest_from_config(CONFIG_PATH, train_dir=TRAIN_DIR, test_dir=TEST_DIR)
print('Source:', manifest['source_id'])
print('Manifest SHA-256:', manifest['manifest_sha256'])
print('Source counts:', manifest['source_counts'])
print('Split counts:', manifest['counts'])
print('Duplicate groups:', manifest['duplicate_audit']['duplicate_groups'])
print('Train/test duplicate groups:', len(manifest['duplicate_audit']['train_test_duplicate_groups']))
print('Conflicting-label groups:', len(manifest['duplicate_audit']['conflicting_label_duplicate_groups']))

The manifest records every source row and exclusion. Same-label training duplicates stay in one split. If an image also appears in source test, its test copy stays there and its training copy is excluded. Conflicting training labels are listed and excluded. No source image or label changes.

## Train the three baselines

`RUN_MODE = 'pilot'` runs one epoch on up to 512 training and 512 validation records. It checks the pipeline; it does not show final model quality. Set it to `'full'` to use the shared configuration: Adam, learning rate 0.001, batch size 64, up to 10 epochs, and validation macro-F1 selection. Both modes call the same `run_experiments` function as the local command. This cell never scores the source test set.

In [ ]:
from src.fer.train import run_experiments

RUN_MODE = 'pilot'  # Change to 'full' for the configured training budget.
if RUN_MODE not in {'pilot', 'full'}:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'.")

common_args = {
    'config_path': CONFIG_PATH,
    'train_dir_override': TRAIN_DIR,
    'test_dir_override': TEST_DIR,
    'manifest_override': MANIFEST_PATH,
}
if RUN_MODE == 'pilot':
    results = run_experiments(
        **common_args, model_ids=['mlp', 'cnn', 'resnet18'], epochs_override=1,
        max_train_records=512, max_validation_records=512,
    )
else:
    results = run_experiments(**common_args, model_ids=['mlp', 'cnn', 'resnet18'])
RUN_DIRECTORY = ROOT / CONFIG['output_root'] / results['run_id']
print('Run:', RUN_DIRECTORY)
print('Run type:', 'bounded pilot' if results['pilot_run'] else 'full-budget validation')
print('Source test evaluated:', results['source_test_evaluated'])

## Review validation results

Validation accuracy and macro-F1 support model comparison. Review per-class scores and confusion matrices too. Keep these results labelled as validation results. Do not select a final model from the 512-record pilot.

In [ ]:
from IPython.display import display, Markdown, Image as NotebookImage
import csv
import matplotlib.pyplot as plt
import numpy as np

comparison_rows = [
    ('Majority reference', 0, 0.0, results['majority_validation_reference']['accuracy'], results['majority_validation_reference']['macro_f1']),
    *[(model_id, item['parameters'], item['training_seconds'], item['validation']['accuracy'], item['validation']['macro_f1'])
      for model_id, item in results['models'].items()],
]
table = ['| Model | Parameters | Training seconds | Validation accuracy | Validation macro-F1 |', '|---|---:|---:|---:|---:|']
table.extend(f'| {name} | {params:,} | {seconds:.1f} | {accuracy:.3f} | {macro_f1:.3f} |'
             for name, params, seconds, accuracy, macro_f1 in comparison_rows)
display(Markdown('\n'.join(table)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (model_id, item) in zip(axes, results['models'].items()):
    matrix = np.asarray(item['validation']['confusion_matrix'])
    axis.imshow(matrix, cmap='Blues')
    axis.set(title=model_id, xlabel='Predicted class', ylabel='Actual class')
    axis.set_xticks(range(7), [name.capitalize() for name in manifest['label_order']], rotation=60, ha='right')
    axis.set_yticks(range(7), [name.capitalize() for name in manifest['label_order']])
fig.tight_layout()
plt.show()

for model_id in results['models']:
    history_path = RUN_DIRECTORY / 'models' / model_id / 'training_history.csv'
    with history_path.open(newline='', encoding='utf-8') as handle:
        history = list(csv.DictReader(handle))
    epochs = [int(row['epoch']) for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].plot(epochs, [float(row['train_loss']) for row in history], label='Train')
    axes[0].plot(epochs, [float(row['validation_loss']) for row in history], label='Validation')
    axes[0].set(title=f'{model_id}: cross-entropy', xlabel='Epoch', ylabel='Loss')
    axes[0].legend()
    axes[1].plot(epochs, [float(row['validation_accuracy']) for row in history], label='Accuracy')
    axes[1].plot(epochs, [float(row['validation_macro_f1']) for row in history], label='Macro-F1')
    axes[1].set(title='Validation scores', xlabel='Epoch', ylabel='Score', ylim=(0, 1))
    axes[1].legend()
    fig.tight_layout()
    plt.show()

## Final test scoring is separate

Do not run this with notebook **Run All**. First review validation results, agree on the model and settings, and keep the selected run unchanged. Then run the separate CLI command from the repository root:

```bash
python -m src.fer.evaluate --checkpoint-dir outputs/runs/<run-id> --split test --confirm-final-test
```

The evaluation command loads the saved checkpoints. It does not create an optimizer, update weights, choose a checkpoint, or train a model. It writes a separate evaluation record under `outputs/evaluations/`.

## Save the run

At the end of Colab, this cell packages the selected run, checkpoints, split manifest, and configuration as one ZIP. It leaves out raw images and `.venv`. Colab downloads the ZIP so the results remain available after the runtime stops. The same export function is available from the local command line.

The ZIP includes a SHA-256 list. Keep it with the report evidence. Do not upload checkpoints to the source repository.

In [ ]:
from src.fer.export_bundle import export_run_bundle

BUNDLE_PATH = ROOT / 'outputs' / 'exports' / f"ft11-fer-{results['run_id']}.zip"
export_run_bundle(RUN_DIRECTORY, config_path=CONFIG_PATH, manifest_path=MANIFEST_PATH, output_path=BUNDLE_PATH)
print('Run bundle:', BUNDLE_PATH)
if IN_COLAB:
    from google.colab import files
    files.download(str(BUNDLE_PATH))

## Sources

Dataset source: KaggleHub `msambare/fer2013/versions/1`, from the [FER2013 image-folder release](https://www.kaggle.com/datasets/msambare/fer2013). Check its access terms. Dataset paper: Goodfellow et al. (2013), [“Challenges in Representation Learning: A report on three machine learning contests”](https://arxiv.org/abs/1307.0414), DOI [10.1007/978-3-642-42051-1_16](https://doi.org/10.1007/978-3-642-42051-1_16).

Course basis: CSCI323 Lecture 2, pages 39–59 and 63; Tutorial 1, pages 28–45; Tutorial 2, pages 3–10. The project instruction files set the report, experiment, contribution, presentation, and Q&A requirements. The exact ResNet18 choice and layer sizes are FT11 choices.